In [ ]:
# Evo-1 LIBERO FP16 Resumable Reference — A100

No quantization. This keeps the Evo-1 server/client split, but avoids one giant fragile client run.

The client runs one suite/task/episode per subprocess and saves every episode log plus a `.done.json` marker to Drive. If Colab stops, rerun the notebook and completed runs are skipped.


In [43]:
# 0. GPU check
import torch, os, subprocess, sys, textwrap, json, re, time
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU")


CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
BF16 supported: True


In [44]:
# 1. Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

BASE        = "/content/drive/MyDrive/cat_baseline"
REPO        = f"{BASE}/Evo-1"
EVO         = f"{REPO}/Evo_1"
LIBERO_EVAL = f"{REPO}/LIBERO_evaluation"
CKPT_DIR    = f"{BASE}/checkpoints/Evo1_LIBERO"
RESULTS     = f"{BASE}/results"
MAMBA       = "/content/micromamba/bin/micromamba"
MAMBA_ROOT  = "/content/micromamba-root"

import os
for p in [BASE, CKPT_DIR, RESULTS]:
    os.makedirs(p, exist_ok=True)
print(BASE, REPO, EVO, LIBERO_EVAL, CKPT_DIR, RESULTS, sep="\n")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/cat_baseline
/content/drive/MyDrive/cat_baseline/Evo-1
/content/drive/MyDrive/cat_baseline/Evo-1/Evo_1
/content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation
/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO
/content/drive/MyDrive/cat_baseline/results


In [45]:
# 2. Clone fresh Evo-1 into cat_baseline — always reset to origin/main
import os
os.makedirs(BASE, exist_ok=True)
%cd "$BASE"
if not os.path.exists(REPO):
    !git clone https://github.com/MINT-SJTU/Evo-1.git Evo-1
else:
    print("Repo exists, resetting to origin/main")

%cd "$REPO"
!git fetch origin
!git reset --hard origin/main
!git clean -fd
!git status --short
print("Repo is clean — no quantization patches applied")


/content/drive/MyDrive/cat_baseline
Repo exists, resetting to origin/main
/content/drive/MyDrive/cat_baseline/Evo-1
HEAD is now at d27d17a update readme
 M .gitignore
 M Evo_1/dataset/config.yaml
 M Evo_1/ds_config.json
 M Evo_1/model/action_head/flow_matching.py
 M Evo_1/scripts/Evo1_server.py
 M MetaWorld_evaluation/mt50_evo1_client_prompt.py
 M MetaWorld_evaluation/tasks.jsonl
 M so100_evo1/lerobot-main/benchmarks/video/capture_camera_feed.py
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/dataset/config.yaml
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/ds_config.json
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/model/action_head/flow_matching.py
 M so100_evo1/lerobot-main/src/lerobot/policies/evo1/scripts/Evo1_server.py
Repo is clean — no quantization patches applied


In [46]:
# 3. Install micromamba and create envs
MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

if not os.path.exists(MAMBA):
    !wget -qO /tmp/micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
    !mkdir -p /content/micromamba
    !tar -xjf /tmp/micromamba.tar.bz2 -C /content/micromamba bin/micromamba

!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n Evo1 python=3.10 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} create -y -n libero python=3.8.13 pip -c conda-forge || true
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} env list


Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.1 sec)

Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /content/micromamba-root/envs/Evo1

  Updating specs:

   - python=3.10
   - pip


  Package               Version  Build                 Channel           Size
─────────────────────────────────────────────────────────────────

In [47]:
# 4. Install Evo-1 server deps
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install -U pip setuptools wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install -r "{EVO}/requirements.txt"
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 python -m pip install --force-reinstall "huggingface-hub==0.36.2"

import subprocess, os

check_code = '''
import transformers, huggingface_hub, torch
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
'''

subprocess.run(
    [MAMBA, "run", "-n", "Evo1", "python", "-c", check_code],
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
    check=True,
)


  Using cached transformers-4.39.0-py3-none-any.whl.metadata (134 kB)
  Using cached timm-1.0.27-py3-none-any.whl.metadata (40 kB)
  Using cached torch-2.5.1-cp310-cp310-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.20.1-cp310-cp310-manylinux1_x86_64.whl.metadata (6.1 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached einop-0.0.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached diffusers-0.38.0-py3-none-any.whl.metadata (20 kB)
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached accelerator-2025.11.11-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (3

CompletedProcess(args=['/content/micromamba/bin/micromamba', 'run', '-n', 'Evo1', 'python', '-c', '\nimport transformers, huggingface_hub, torch\nprint("transformers:", transformers.__version__)\nprint("huggingface_hub:", huggingface_hub.__version__)\nprint("torch:", torch.__version__)\nprint("CUDA:", torch.cuda.is_available())\n'], returncode=0)

In [48]:
# 5. A100: try flash-attn, but do not force-disable it
!cd "{EVO}" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n Evo1 bash -lc 'MAX_JOBS=4 pip install -v flash-attn --no-build-isolation' || echo "flash-attn install failed/skipped"


Using pip 26.1.1 from /content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/pip (python 3.10)
  Using cached flash_attn-2.8.3-cp310-cp310-linux_x86_64.whl


In [49]:
# 6. Install LIBERO env
!cd "{LIBERO_EVAL}" && test -d LIBERO || git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -U "pip<25.1" "setuptools<76" wheel
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install "numpy<1.24" "protobuf<4"
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -r requirements.txt
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
!cd "{LIBERO_EVAL}/LIBERO" && MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install -e .
!MAMBA_ROOT_PREFIX={MAMBA_ROOT} {MAMBA} run -n libero python -m pip install websockets==13.1 huggingface_hub imageio imageio-ffmpeg opencv-python


  Using cached pip-25.0.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached setuptools-75.3.4-py3-none-any.whl.metadata (6.9 kB)
Using cached pip-25.0.1-py3-none-any.whl (1.8 MB)
Using cached setuptools-75.3.4-py3-none-any.whl (1.3 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.3.0
    Uninstalling setuptools-75.3.0:
      Successfully uninstalled setuptools-75.3.0
  Attempting uninstall: pip
    Found existing installation: pip 24.3.1
    Uninstalling pip-24.3.1:
      Successfully uninstalled pip-24.3.1
  Using cached numpy-1.23.5-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
  Using cached protobuf-3.20.3-cp38-cp38-manylinux_2_5_x86_64.manylinux1_x86_64.whl.metadata (679 bytes)
Using cached numpy-1.23.5-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
Using cached protobuf-3.20.3-cp38-cp38-manylinux_2_5_x86_64.manylinux1_x86_64.whl (1.0 MB)
  Using cached hydra_core-1.2.0-py3-none-any.whl.metadata 

In [50]:
# 7. Download checkpoint
from huggingface_hub import snapshot_download
import os
os.makedirs(CKPT_DIR, exist_ok=True)
snapshot_download(repo_id="MINT-SJTU/Evo1_LIBERO", local_dir=CKPT_DIR, local_dir_use_symlinks=False)
!find "{CKPT_DIR}" -maxdepth 2 -type f | sort | head -50


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO/checkpoint.json
/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO/config.json
/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO/mp_rank_00_model_states.pt
/content/drive/MyDrive/cat_baseline/checkpoints/Evo1_LIBERO/norm_stats.json


In [51]:
# 8. Patch server: checkpoint path, port 9010, websocket no-timeout
from pathlib import Path
import re

server_path = Path(f"{EVO}/scripts/Evo1_server.py")
txt = server_path.read_text()

txt = re.sub(
    r"ckpt_dir\s*=\s*['\"].*?['\"]",
    f'ckpt_dir = "{CKPT_DIR}"',
    txt,
    count=1,
)

txt = txt.replace("9000", "9010")

if "ping_interval=None" not in txt:
    txt = txt.replace(
        'websockets.serve(handler, "0.0.0.0", PORT)',
        'websockets.serve(handler, "0.0.0.0", PORT, ping_interval=None, ping_timeout=None, close_timeout=30)',
    )
    txt = txt.replace(
        "websockets.serve(handler, '0.0.0.0', PORT)",
        "websockets.serve(handler, '0.0.0.0', PORT, ping_interval=None, ping_timeout=None, close_timeout=30)",
    )

server_path.write_text(txt)
!grep -n "ckpt_dir|9010|9000|websockets.serve|ping_interval" "{server_path}" | tail -40


In [52]:
# 9. LIBERO config
import os

os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
os.makedirs(f'{LIBERO_EVAL}/LIBERO/libero/datasets', exist_ok=True)

config_yaml = (
    f'benchmark_root: {LIBERO_EVAL}/LIBERO/libero/libero\n'
    f'bddl_files: {LIBERO_EVAL}/LIBERO/libero/libero/bddl_files\n'
    f'init_states: {LIBERO_EVAL}/LIBERO/libero/libero/init_files\n'
    f'datasets: {LIBERO_EVAL}/LIBERO/libero/datasets\n'
    f'assets: {LIBERO_EVAL}/LIBERO/libero/libero/assets\n'
)

config_path = os.path.expanduser('~/.libero/config.yaml')
with open(config_path, 'w') as f:
    f.write(config_yaml)

print(open(config_path).read())


benchmark_root: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero
bddl_files: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/bddl_files
init_states: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/init_files
datasets: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/datasets
assets: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/LIBERO/libero/libero/assets



In [64]:
# 10. Create runtime single-episode LIBERO client copy
from pathlib import Path
import re

src = Path(f'{LIBERO_EVAL}/libero_client_4tasks.py')
dst = Path(f'{LIBERO_EVAL}/libero_client_single_episode_runtime.py')

original = src.read_text()

prefix = """
import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------
"""

txt = prefix + '\n' + original

txt = re.sub(r"SERVER_URL\s*=\s*['\"].*?['\"]", 'SERVER_URL = "ws://127.0.0.1:9010"', txt, count=1)
txt = re.sub(r'horizon\s*=\s*\d+', 'horizon = 14', txt, count=1)
txt = re.sub(r'max_steps\s*=\s*\[[^\]]+\]', 'max_steps = [SINGLE_MAX_STEPS]', txt, count=1)
txt = re.sub(r'task_suites\s*=\s*\[[^\]]+\]', 'task_suites = [SINGLE_SUITE]', txt, count=1)
txt = re.sub(r'num_episodes\s*=\s*\d+', 'num_episodes = 1', txt, count=1)
txt = re.sub(r"ckpt_name\s*=\s*f?['\"].*?['\"]", 'ckpt_name = SINGLE_CKPT_NAME', txt, count=1)

txt = txt.replace('for task_id in range(num_tasks_in_suite):', 'for task_id in [SINGLE_TASK_ID]:')
txt = txt.replace('for task_id in range(min(num_tasks_in_suite, 1)):', 'for task_id in [SINGLE_TASK_ID]:')
txt = txt.replace('for task_id in range(min(num_tasks_in_suite, 10)):', 'for task_id in [SINGLE_TASK_ID]:')
txt = txt.replace('initial_states[episode_id]', 'initial_states[SINGLE_EP_INDEX]')
txt = txt.replace('init_states[episode_id]', 'init_states[SINGLE_EP_INDEX]')
txt = txt.replace('for episode_id in range(num_episodes):', 'for episode_id in [0]:')
txt = txt.replace(
    'async with websockets.connect(SERVER_URL) as ws:',
    'async with websockets.connect(SERVER_URL, max_size=100_000_000, ping_interval=None, ping_timeout=None, close_timeout=30) as ws:',
)

dst.write_text(txt)
print('Wrote:', dst)
print('\nTop of generated runtime client:')
print('\n'.join(dst.read_text().splitlines()[:20]))

# Remove stale debug markers
for name in [
    'fp16_baseline_debug_libero_spatial_task0_ep0.log',
    'fp16_baseline_debug_libero_spatial_task0_ep0.done.json',
]:
    p = Path(RESULTS) / name
    if p.exists():
        p.unlink()
        print('deleted stale file:', p)


Wrote: /content/drive/MyDrive/cat_baseline/Evo-1/LIBERO_evaluation/libero_client_single_episode_runtime.py

Top of generated runtime client:

import os

# ---- Runtime single-episode controls inserted by Colab notebook ----
SINGLE_TASK_ID = int(os.environ.get("SINGLE_TASK_ID", "0"))
SINGLE_EP_INDEX = int(os.environ.get("SINGLE_EP_INDEX", "0"))
SINGLE_SUITE = os.environ.get("SINGLE_SUITE", "libero_spatial")
SINGLE_MAX_STEPS = int(os.environ.get("SINGLE_MAX_STEPS", "25"))
SINGLE_CKPT_NAME = os.environ.get(
    "SINGLE_CKPT_NAME",
    f"Evo1_FP16_{SINGLE_SUITE}_task{SINGLE_TASK_ID}_ep{SINGLE_EP_INDEX}",
)
# -------------------------------------------------------------------

import asyncio
import websockets
import numpy as np
import json
import pathlib
import os


In [65]:
# Fast local checkpoint copy + server restart
import subprocess, os, re, time
from pathlib import Path

LOCAL_CKPT = '/content/Evo1_LIBERO'
SERVER_LOG = '/content/evo1_fp16_server.log'

# Copy checkpoint from Drive to local Colab disk
subprocess.run(['mkdir', '-p', LOCAL_CKPT], check=True)
subprocess.run(['rsync', '-ah', '--info=progress2', CKPT_DIR + '/', LOCAL_CKPT + '/'], check=True)

# Patch server to use local checkpoint path
server_path = Path(f'{EVO}/scripts/Evo1_server.py')
txt = server_path.read_text()
txt = re.sub(
    r"ckpt_dir\s*=\s*['\"].*?['\"]",
    f'ckpt_dir = "{LOCAL_CKPT}"',
    txt,
    count=1,
)
txt = txt.replace('9000', '9010')
if 'ping_interval=None' not in txt:
    txt = txt.replace(
        'websockets.serve(handler, "0.0.0.0", PORT)',
        'websockets.serve(handler, "0.0.0.0", PORT, ping_interval=None, ping_timeout=None, close_timeout=30)',
    )
server_path.write_text(txt)

# Restart server
subprocess.run(['pkill', '-f', 'Evo1_server.py'], check=False)

server_proc = subprocess.Popen(
    [MAMBA, 'run', '-n', 'Evo1', 'python', '-u', 'scripts/Evo1_server.py'],
    cwd=EVO,
    stdout=open(SERVER_LOG, 'w'),
    stderr=subprocess.STDOUT,
    env={**os.environ, 'MAMBA_ROOT_PREFIX': MAMBA_ROOT},
)

print('server pid:', server_proc.pid)
time.sleep(20)
print(Path(SERVER_LOG).read_text(errors='ignore').splitlines()[-80:])


server pid: 274119
['Loading EVO_1 model...', '/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.', '  warnings.warn(', 'Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.', 'Traceback (most recent call last):', '  File "/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/transformers/utils/import_utils.py", line 1472, in _get_module', '    return importlib.import_module("." + module_name, self.__name__)', '  File "/content/micromamba-root/envs/Evo1/lib/python3.10/importlib/__init__.py", line 126, in import_module', '    return _bootstrap._gcd_import(name[level:], package, level)', '  File "<frozen importlib._bootstrap>", line 1050, in _gcd_import', '  File "<frozen importl

In [66]:
!ps -ef | grep Evo1_server.py | grep -v grep || echo "SERVER DEAD"
!ss -ltnp | grep 9010 || echo "PORT 9010 NOT OPEN"
!nvidia-smi --query-gpu=name,memory.used,memory.free,utilization.gpu --format=csv,noheader
!tail -n 120 /content/evo1_fp16_server.log || true

SERVER DEAD
LISTEN 0      100          0.0.0.0:9010       0.0.0.0:*    users:(("python",pid=45478,fd=58))      
NVIDIA RTX PRO 6000 Blackwell Server Edition, 2967 MiB, 94284 MiB, 0 %
Loading EVO_1 model...
/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Traceback (most recent call last):
  File "/content/micromamba-root/envs/Evo1/lib/python3.10/site-packages/transformers/utils/import_utils.py", line 1472, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
  File "/content/micromamba-root/envs/Evo1/lib/python3.10/importlib/__init__.py", line 126, in import_module
    return _bootstrap

In [67]:
# 13. Websocket-only test — pure Python version

import subprocess, os

MAMBA = "/content/micromamba/bin/micromamba"
MAMBA_ROOT = "/content/micromamba-root"

ws_test_code = r'''
import asyncio
import websockets

async def main():
    url = "ws://127.0.0.1:9010"
    print("trying", url)
    async with websockets.connect(
        url,
        max_size=100_000_000,
        ping_interval=None,
        ping_timeout=None,
        close_timeout=30,
    ) as ws:
        print("CONNECTED OK")

asyncio.run(main())
'''

subprocess.run(
    [MAMBA, "run", "-n", "libero", "python", "-c", ws_test_code],
    env={**os.environ, "MAMBA_ROOT_PREFIX": MAMBA_ROOT},
    check=True,
)

CompletedProcess(args=['/content/micromamba/bin/micromamba', 'run', '-n', 'libero', 'python', '-c', '\nimport asyncio\nimport websockets\n\nasync def main():\n    url = "ws://127.0.0.1:9010"\n    print("trying", url)\n    async with websockets.connect(\n        url,\n        max_size=100_000_000,\n        ping_interval=None,\n        ping_timeout=None,\n        close_timeout=30,\n    ) as ws:\n        print("CONNECTED OK")\n\nasyncio.run(main())\n'], returncode=0)

In [68]:
# FP16 eval settings — 4 suites x 10 tasks x 10 episodes, resumable
import json, time
from pathlib import Path

TASK_SUITES = ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
TASK_IDS = list(range(10))
EPISODES = list(range(10))
EXPECTED_EVAL_RUNS = len(TASK_SUITES) * len(TASK_IDS) * len(EPISODES)

MAX_STEPS_BY_SUITE = {
    'libero_spatial': 25,
    'libero_object': 25,
    'libero_goal': 25,
    'libero_10': 95,
}

TAG = 'fp16_baseline'
RESULTS_PATH = Path(RESULTS)
EVAL_RESULTS = RESULTS_PATH / 'eval_4suites_10ep'
EVAL_RESULTS.mkdir(parents=True, exist_ok=True)

manifest = {
    'tag': TAG,
    'task_suites': TASK_SUITES,
    'task_ids': TASK_IDS,
    'episodes': EPISODES,
    'expected_eval_runs': EXPECTED_EVAL_RUNS,
    'max_steps_by_suite': MAX_STEPS_BY_SUITE,
    'success_rule': 'episode log contains checkmark Success only',
    'quant': 'none_fp16',
    'created_time': time.time(),
}
(EVAL_RESULTS / f'{TAG}_benchmark_manifest.json').write_text(json.dumps(manifest, indent=2))

print('TAG:', TAG)
print('EVAL_RESULTS:', EVAL_RESULTS)
print('TASK_SUITES:', TASK_SUITES)
print('EXPECTED_EVAL_RUNS:', EXPECTED_EVAL_RUNS)
print('MAX_STEPS_BY_SUITE:', MAX_STEPS_BY_SUITE)
print('RESUME_RULE: completed episodes have *.done.json and will be skipped on rerun.')


TAG: fp16_baseline
EVAL_RESULTS: /content/drive/MyDrive/cat_baseline/results/eval_4suites_10ep
TASK_SUITES: ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']
EXPECTED_EVAL_RUNS: 400
MAX_STEPS_BY_SUITE: {'libero_spatial': 25, 'libero_object': 25, 'libero_goal': 25, 'libero_10': 95}
RESUME_RULE: completed episodes have *.done.json and will be skipped on rerun.


In [69]:
# FP16 run loop — per-task and per-suite results, overall result, accuracy saved
import subprocess, os, time, json
from pathlib import Path

def server_listening():
    r = subprocess.run("ss -ltnp | grep '9010'", shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0

def run_one(suite, task_id, ep):
    name = f'{TAG}_{suite}_task{task_id}_ep{ep}'
    log = EVAL_RESULTS / f'{name}.log'
    done = EVAL_RESULTS / f'{name}.done.json'
    if done.exists():
        try:
            prev = json.loads(done.read_text())
            result = 'SUCCESS' if bool(prev.get('success')) else 'FAIL'
            crash_prev = bool(prev.get('server_crash', prev.get('crash', False)))
            print(f'EPISODE_SKIP suite={suite} task={task_id:02d} ep={ep:02d} already_done ({result}) crash={crash_prev}', flush=True)
        except Exception as exc:
            print(f'EPISODE_SKIP suite={suite} task={task_id:02d} ep={ep:02d} already_done (unreadable: {exc})', flush=True)
        return
    if not server_listening():
        raise RuntimeError('Server not listening on 9010')
    env = {**os.environ, 'MAMBA_ROOT_PREFIX': MAMBA_ROOT, 'SINGLE_SUITE': suite,
           'SINGLE_TASK_ID': str(task_id), 'SINGLE_EP_INDEX': str(ep),
           'SINGLE_MAX_STEPS': str(MAX_STEPS_BY_SUITE[suite]), 'SINGLE_CKPT_NAME': name}
    with open(log, 'w') as f:
        p = subprocess.run([MAMBA, 'run', '-n', 'libero', 'python', '-u', 'libero_client_single_episode_runtime.py'],
                           cwd=LIBERO_EVAL, env=env, stdout=f, stderr=subprocess.STDOUT, text=True)
    text = log.read_text(errors='ignore')
    egl = ('EGL_NOT_INITIALIZED' in text or 'EGLGLContext.__del__' in text)
    crash = (p.returncode != 0) or ('Traceback' in text and not (p.returncode == 0 and egl))
    success = ('✅ Success' in text)
    rec = {'suite': suite, 'task_id': task_id, 'episode': ep, 'returncode': p.returncode,
           'success': success, 'task_fail': (not success and not crash), 'server_crash': crash,
           'egl_cleanup_warning': egl, 'log_path': str(log), 'time': time.time()}
    done.write_text(json.dumps(rec, indent=2))
    print(f'EPISODE_RESULT suite={suite} task={task_id:02d} ep={ep:02d} ' + ('SUCCESS' if success else 'FAIL') + f' crash={crash}', flush=True)
    if crash or not success:
        print('EPISODE_LOG_TAIL_BEGIN')
        print('\n'.join(text.splitlines()[-25:]))
        print('EPISODE_LOG_TAIL_END')

overall_done = 0
overall_success = 0
run_suite_results = {}
run_task_results = {}
for s in TASK_SUITES:
    suite_done = 0
    suite_success = 0
    run_task_results[s] = {}
    for t in TASK_IDS:
        for e in EPISODES:
            run_one(s, t, e)
        task_records = []
        for e in EPISODES:
            p = EVAL_RESULTS / f'{TAG}_{s}_task{t}_ep{e}.done.json'
            if p.exists():
                try:
                    task_records.append(json.loads(p.read_text()))
                except Exception as exc:
                    print(f'TASK_RESULT_READ_WARN suite={s} task={t:02d} ep={e:02d} {exc}', flush=True)
        wins = sum(1 for r in task_records if bool(r.get('success')))
        n = len(task_records)
        suite_done += n
        suite_success += wins
        run_task_results[s][t] = {'success': wins, 'total': n, 'rate': wins / n if n else 0}
        print(f'TASK_RESULT suite={s} task={t:02d} success={wins}/{n} rate={(wins / n if n else 0):.3f}', flush=True)
    run_suite_results[s] = {'success': suite_success, 'total': suite_done, 'rate': suite_success / suite_done if suite_done else 0}
    overall_done += suite_done
    overall_success += suite_success
    print(f'SUITE_PROGRESS suite={s} success={suite_success}/{suite_done} rate={(suite_success / suite_done if suite_done else 0):.3f}', flush=True)

overall_rate = overall_success / overall_done if overall_done else 0
print(f'OVERALL_RESULT success={overall_success}/{overall_done} rate={overall_rate:.3f}', flush=True)
acc_record = {
    'tag': TAG,
    'quant': 'none_fp16',
    'overall': {'success': overall_success, 'total': overall_done, 'rate': overall_rate},
    'by_suite': run_suite_results,
    'by_task': {s: {str(t): v for t, v in tasks.items()} for s, tasks in run_task_results.items()},
}
ACC_PATH = EVAL_RESULTS / f'{TAG}_run_accuracy.json'
ACC_PATH.write_text(json.dumps(acc_record, indent=2))
print(f'ACCURACY_SAVED: {ACC_PATH}', flush=True)


EPISODE_RESULT suite=libero_spatial task=00 ep=00 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=01 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=02 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=03 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=04 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=05 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=06 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=07 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=08 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=00 ep=09 SUCCESS crash=False
TASK_RESULT suite=libero_spatial task=00 success=10/10 rate=1.000
EPISODE_RESULT suite=libero_spatial task=01 ep=00 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=01 ep=01 SUCCESS crash=False
EPISODE_RESULT suite=libero_spatial task=01 ep=02 SUCCESS crash=False
EPISODE_RESULT suite=lib

In [70]:
# 16. Combine per-episode summaries
import json
from pathlib import Path
import pandas as pd

records = []
for p in sorted(EVAL_RESULTS.glob(f'{TAG}_*.done.json')):
    records.append(json.loads(p.read_text()))

df = pd.DataFrame(records)
if len(df) == 0:
    print('No done summaries yet.')
else:
    display(df)
    agg = df.groupby('suite').agg(
        runs=('log_path', 'count'),
        success=('success', 'sum'),
        crashed=('server_crash', 'sum'),
    )
    agg['success_rate'] = agg['success'] / agg['runs']
    display(agg)

    out_csv = EVAL_RESULTS / f'{TAG}_combined.csv'
    out_json = EVAL_RESULTS / f'{TAG}_combined.json'
    df.to_csv(out_csv, index=False)
    out_json.write_text(json.dumps(records, indent=2))
    print('Saved:', out_csv)
    print('Saved:', out_json)


,suite,task_id,episode,returncode,success,task_fail,server_crash,egl_cleanup_warning,log_path,time
0,libero_10,0,0,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778851e+09
1,libero_10,0,1,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778851e+09
2,libero_10,0,2,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778851e+09
3,libero_10,0,3,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778851e+09
4,libero_10,0,4,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778851e+09
...,...,...,...,...,...,...,...,...,...,...
395,libero_spatial,9,5,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778845e+09
396,libero_spatial,9,6,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778845e+09
397,libero_spatial,9,7,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778845e+09
398,libero_spatial,9,8,0,True,False,False,True,/content/drive/MyDrive/cat_baseline/results/ev...,1.778845e+09


,runs,success,crashed,success_rate
suite,,,,
libero_10,100,89,0,0.89
libero_goal,100,95,0,0.95
libero_object,100,94,0,0.94
libero_spatial,100,94,0,0.94


Saved: /content/drive/MyDrive/cat_baseline/results/eval_4suites_10ep/fp16_baseline_combined.csv
Saved: /content/drive/MyDrive/cat_baseline/results/eval_4suites_10ep/fp16_baseline_combined.json


In [ ]:
# # 17. Stop server when finished
# !pkill -f Evo1_server.py || true
# !ps -ef | grep Evo1_server.py | grep -v grep || echo "SERVER STOPPED"


## Expand after debug

After `task0 × episode0` works, edit cell 14.

Spatial reference:

```python
TASK_SUITES = ["libero_spatial"]
TASK_IDS = list(range(10))
EPISODES = list(range(5))
TAG = "fp16_spatial_10tasks_5ep"
```

Full reference:

```python
TASK_SUITES = ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
TASK_IDS = list(range(10))
EPISODES = list(range(10))
TAG = "fp16_full_4suites_10ep"
```
